# Heat equation - PDE viewpoint: order and convergence

In [ ]:
#    APM41012EP course notebook - Chapter 9 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Heat equation - PDE viewpoint: order and convergence
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
from dataclasses import dataclass
import numpy as np
from scipy.integrate import solve_ivp
from scipy.sparse import diags
from scipy import sparse
from scipy.sparse.linalg import spsolve
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Heat equation

We consider the heat equation in 1D:

$$
\left\{
\begin{aligned}
& \partial_t u(x,t) - D \, \partial_{xx} u(x,t) = 0 \;\; \text{for} \; -L < x < L\\
& u(x,0)  =  \delta_0(x),
\end{aligned}
\right.
$$

where $\delta_0(x)$ denotes the Dirac delta function at the origin $x=0$. We take $L=5$ and $D=1$, with homogeneous Neumann boundary conditions.

The analytical solution of the elementary heat equation on the whole real line reads:

\begin{equation}
u(x,t) = \frac{1}{2 \sqrt{\pi D t}} \exp \Bigg(-\frac{x^2}{\sqrt{4Dt}} \Bigg)
\end{equation}

The order studies that follow are carried out by comparing the numerical solution (for which both the space and time steps tend to zero - in the PDE sense) with the restriction of the exact solution to the interval $[-L,L]$, where $L$ has been taken large enough for the influence of the boundary conditions to be negligible.

In [ ]:
#############################################
class heat_model:

    def __init__(self, d, xmin, xmax, nx) :
        self.d = d
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx-1)
        self.oneoverdxdx = 1/(self.dx*self.dx)
        
        doverdxdx = d/(self.dx**2)
        diag = np.repeat(-2, nx)
        sub_diag = np.repeat([1., 2.], [nx-2, 1])
        self.a = doverdxdx * diags([diag, sub_diag, np.flip(sub_diag)], [0, -1, 1])

    def fcn(self, t, y):
        return self.a.dot(y)
    
    def fcn_exact(self, t):
        xmin = self.xmin
        xmax = self.xmax
        d = self.d
        nx = self.nx
        dx = self.dx
        x = np.linspace(xmin, xmax, nx)
        y = (1/(2*np.sqrt(np.pi*t))) * np.exp(-(x*x)/(4.*t))
        return y

## Explicit Euler method

In this case, the stability limit of the method imposes $\Delta t \le \frac{1}{2}(\Delta x)^2$.

To limit the computational cost, we therefore take exactly a limit in which the space step tends to zero with
$\Delta t = \frac{1}{2}(\Delta x)^2$.

The error can be plotted as a function of the space step (order two) or as a function of the time step (order 1), but here it is generally the space step that drives the error, whose spatial and temporal parts are of the same order of magnitude.

In [ ]:
@dataclass
class ode_result:
    t: np.ndarray 
    y: np.ndarray    
    
#############################################
def forward_euler(tini, tend, nt, yini, fcn):
    
    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    for it, tn  in enumerate(t[:-1]):
        yn = yn + dt*fcn(tn, yn)
        
    return ode_result(t, yn)

#################################################
def plot_fe_error(d, tini, tend, xmin, xmax, nx):
    
    l_dx = []
    l_dt = []
    l_err_exa = [] 

    for nxi in nx:

        dx = (xmax-xmin)/(nxi-1)

        hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nxi)
        fcn_exact  = hm.fcn_exact
        fcn = hm.fcn

        # initial solution 
        yini = hm.fcn_exact(tini)

        # exact solution 
        yexa = hm.fcn_exact(tend)

        dt = (dx**2)/2
        nt = int((tend-tini)/dt) + 1

        # forward euler integration
        sol = forward_euler(tini, tend, nt, yini, fcn)
        ysol = sol.y

        # error
        err_exa = np.abs(yexa-ysol)
        norm_err_exa = np.linalg.norm(err_exa) / np.sqrt(nxi)
        ##print(f"l2 norm of the error for dt = {dt:.3e} : {norm_err_exa:.10e}")

        l_dx.append(dx)
        l_dt.append(dt)
        l_err_exa.append(norm_err_exa)

    fig_dx = go.Figure()
    fig_dx.add_trace(go.Scatter(x=l_dx, y=l_err_exa, name='error'))
    fig_dx.add_trace(go.Scatter(x=l_dx, y=np.array(l_dx)**2, name='slope 2'))
    fig_dx.update_xaxes(type='log', exponentformat='e', title='dx')
    fig_dx.update_yaxes(type='log', exponentformat='e', title='error')
    fig_dx.update_layout(height=500, title='PDE error: dt = 1/2(dx)^2')
    fig_dx.show()

    fig_dt = go.Figure()
    fig_dt.add_trace(go.Scatter(x=l_dt, y=l_err_exa, name='error'))
    fig_dt.add_trace(go.Scatter(x=l_dt, y=l_dt, name='slope 1'))
    fig_dt.update_xaxes(type='log', exponentformat='e', title='dt')
    fig_dt.update_yaxes(type='log', exponentformat='e', title='error')
    fig_dt.update_layout(height=500, title='PDE error')
    fig_dt.show()

In [ ]:
d = 1.

tini = 0.01
tend = 0.11


xmin = -5.
xmax = 5.

# nx for dt = 0.5 dx.dx
nx = np.array([1001, 2001, 4001, 8001, 10001, 16001])

# computation and display of the error as a function of dx and dt
plot_fe_error(d, tini, tend, xmin, xmax, nx)

## Crank-Nicolson method

In this case, since there is no stability limit, we take two limits in which the space step tends to zero: with $\Delta t = (\Delta x)^2$ (the temporal error is then negligible compared with the spatial error), where we recover the previous results (order two in space and one in time), but also with  $\Delta t = \frac{1}{2}\Delta x$, which yields spatial and temporal errors of the same order of magnitude. In this case,
we obtain order two in space and two in time.

In [ ]:
#############################################
def crank_nicolson(tini, tend, nt, yini, a):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    cn_mat = sparse.eye(yini.size) - 0.5*dt*a

    for it, tn  in enumerate(t[:-1]):
        yn = spsolve(cn_mat.tocsr(), yn + 0.5*dt*a.dot(yn))

    return ode_result(t, yn)

#######################################################
def plot_cn_error(d, tini, tend, xmin, xmax, nx1, nx2):

    l_dx1 = []
    l_dt1 = []
    l_err_exa1 = [] 

    for nxi in nx1:

        dx = (xmax-xmin)/(nxi)
        
        hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nxi)
        fcn_exact  = hm.fcn_exact
        fcn = hm.fcn
        a = hm.a

        # initial solution 
        yini = hm.fcn_exact(tini)

        # exact solution 
        yexa = hm.fcn_exact(tend)

        dt = dx**2 
        nt = int((tend-tini)/dt) + 1

        # crank-nicolson  integration
        sol = crank_nicolson(tini, tend, nt, yini, a)
        ysol = sol.y

        # error
        err_exa = np.abs(yexa-ysol)
        norm_err_exa = np.linalg.norm(err_exa) / np.sqrt(nxi)
        ##print(f"l2 norm of the error for dt = {dt:.3e} : {norm_err_exa:.10e}")

        l_dx1.append(dx)
        l_dt1.append(dt)
        l_err_exa1.append(norm_err_exa)


    l_dx2 = []
    l_dt2 = []
    l_err_exa2 = [] 

    for nxi in nx2:
        
        dx = (xmax-xmin)/(nxi-1)
        
        hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nxi)
        fcn_exact  = hm.fcn_exact
        fcn = hm.fcn
        a = hm.a

        # initial solution 
        yini = hm.fcn_exact(tini)

        # exact solution 
        yexa = hm.fcn_exact(tend)

        dt = dx / 3
        nt = int((tend-tini)/dt) + 1
        
        # crank-nicolson  integration
        sol = crank_nicolson(tini, tend, nt, yini, a)
        ysol = sol.y

        # error
        err_exa = np.abs(yexa-ysol)
        norm_err_exa = np.linalg.norm(err_exa) / np.sqrt(nxi)
        ##print(f"l2 norm of the error for dt = {dt:.3e} : {norm_err_exa:.10e}")

        l_dx2.append(dx)
        l_dt2.append(dt)
        l_err_exa2.append(norm_err_exa)

    fig_dx = go.Figure()
    fig_dx.add_trace(go.Scatter(x=l_dx1, y=l_err_exa1, name='error (dt=dx.dx)'))
    fig_dx.add_trace(go.Scatter(x=l_dx1, y=np.array(l_dx1)**2, name='slope 2'))
    fig_dx.add_trace(go.Scatter(x=l_dx2, y=l_err_exa2, name='error (dt=dx/3)'))
    fig_dx.update_xaxes(type='log', exponentformat='e', title='dx')
    fig_dx.update_yaxes(type='log', exponentformat='e', title='error')
    fig_dx.update_layout(height=500, title='PDE error')
    fig_dx.show()

    fig_dt = go.Figure()
    fig_dt.add_trace(go.Scatter(x=l_dt1, y=l_err_exa1, name='error (dt=dx.dx)'))
    fig_dt.add_trace(go.Scatter(x=l_dt1, y=l_dt1, name='slope 1'))
    fig_dt.add_trace(go.Scatter(x=l_dt2, y=l_err_exa2, name='error (dt=dx/3)'))
    fig_dt.add_trace(go.Scatter(x=l_dt2, y=np.array(l_dt2)**2, name='slope 2', ))

    fig_dt.update_xaxes(type='log', exponentformat='e', title='dt')
    fig_dt.update_yaxes(type='log', exponentformat='e', title='error')
    fig_dt.update_layout(height=500, title='PDE error')
    fig_dt.show()

In [ ]:
d = 1.    

tini = 0.01
tend = 0.11

xmin = -5.
xmax = 5.

# nx for the case dt = dx.dx
nx1 = np.array([299, 599, 999, 1999, 3999, 5999, 7999])

# nx for the case dt = dx/3
nx2 = np.array([599, 999, 1999, 3999, 5999, 7999, 9999, 15999, 19999])

# computation and display of the error as a function of dt and dx (warning: long computation...)
plot_cn_error(d, tini, tend, xmin, xmax, nx1, nx2)